In [ ]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, ModelSettings, WebSearchTool
import resend
import asyncio
from agents.extensions.visualization import draw_graph
from IPython.display import display, Markdown
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

# Runner.run

In [ ]:
batman_agent = Agent(name="Batman", instructions="Imagine you are Batman. You reply like him, talk like him and act like him. ", model="gpt-5.4-mini")
result = await Runner.run(batman_agent, "Tell a joke about Autonomous AI Agents")

# Trace

In [ ]:
with trace("Telling a joke"):
    result = await Runner.run(batman_agent, "Do you work during the day?")
print(result.final_output)

# Streaming

In [ ]:
result = Runner.run_streamed(batman_agent, input="Tell me 3 accomplishments of yours.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

In [ ]:
@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [ ]:
notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user of any replies.", tools=[push_tool])


with trace("Mail notification"):
    result = await Runner.run(notifier, "Client replied: Thanks for the update, can you also share the cost estimations?")

print(result.final_output)

# Sessions and memory

In [ ]:
MODEL = "gpt-5.4-mini"

In [ ]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [ ]:
response = await Runner.run(agent, "Hi there. I work at Dcastalia.")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "Where do I work")
print(response.final_output) 

### Approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. I work at Dcastalia.")
print(response.final_output)

response.to_input_list()

next_input = response.to_input_list() + [{"role": "user", "content": "Where do I work?"}]
next_input

response = await Runner.run(agent, next_input)
print(response.final_output)

### Approach 2 - OpenAI Agents SDK built in SQLLite session

In [ ]:
session = SQLiteSession("12346")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Jawahiir.", session=session)
print(response.final_output)

response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

# Resend email setup

In [ ]:
RESEND_API_KEY = os.getenv("RESEND_API_KEY")
sender = os.getenv("RESEND_FROM")
receiver = os.getenv("RESEND_TO")
resend.api_key = RESEND_API_KEY 

In [ ]:
@function_tool
def send_email_tool(subject: str, body: str, to: str = "") -> str:
    """Send an email to the user.

    Args:
        subject: Email subject line.
        body: Plain-text email body.
        to: Recipient address; defaults to RESEND_TO from .env when empty.
    """
    if not RESEND_API_KEY:
        return "Error: RESEND_API_KEY is not set - check repo-root .env."
    if not sender:
        return "Error: RESEND_FROM is not set in .env."
    recipient = (to or receiver).strip()
    if not recipient or "@" not in recipient:
        return f"Error: invalid recipient {recipient!r} - set RESEND_TO in .env."
    try:
        resp = resend.Emails.send({
            "from": sender,
            "to": [recipient],
            "subject": subject,
            "text": body,
        })
    except Exception as e: 
        return f"Error sending email: {e}"
    email_id = resp.get("id") if isinstance(resp, dict) else getattr(resp, "id", resp)
    return f"Email sent to {recipient} (id={email_id})"

In [ ]:
MODEL_NAME = "gpt-5.4-mini"

# Agent Orchestration

In [ ]:
intro = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

### Orchestration by code

In [ ]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)

In [ ]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required")
 
sales_sender = Agent(name="Sales Sender", instructions=decision, model=MODEL_NAME, tools=[send_email_tool], model_settings=require_tool)

In [ ]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

### Orchestration via LLM

via Tools
The simplest way to have 1 Agent choose to invoke another is by treating it as a tool call.

The OpenAI Agents SDK gives a very simple way to do this.

This works best when the flow is:

Agent A -> Agent B -> Agent A

And for the classic "Planning Agent" situation.

In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=MODEL_NAME)

In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

### via Handoffs
Very unreliable. They're not used consistently by other frameworks.

Behind the scenes, OpenAI Agents SDK has implemented these with Tools anyway.

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. You get your sales team to draft emails, then send them all to a sales picker.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Handoff to the sales sender to choose and send the best email.
"""

tools = [tool1, tool2, tool3]
handoffs = [sales_sender]

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, handoffs=handoffs, model=MODEL_NAME)

In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

# OpenAI Hosted Tools

In [ ]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

### Agent 1: The Search Agent

In [ ]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Tell me about Dcastalia Limited."
 
settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [ ]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))